In [26]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [27]:
df = pd.read_csv(
    "../data/processed/cardiac_validated.csv"
)

In [28]:
X = df.drop(columns=["patient_id", "cardiac_risk"])
y = df["cardiac_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [29]:
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
numeric_features_model = X.select_dtypes(include=np.number).columns.tolist()

In [34]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features_model),
    ('categorical', categorical_pipeline, categorical_features),
])


baseline =  Pipeline([
        ('preprocess', preprocessor),
        ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    ])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}
scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

record = {
    'model': "Logistic Regression"
}

for metric in scoring:
    record[f'cv_{metric}_mean'] = scores[f'test_{metric}'].mean()
    record[f'cv_{metric}_std'] = scores[f'test_{metric}'].std()

record_df = pd.DataFrame([record])

display(record_df.round(3))

,model,cv_accuracy_mean,cv_accuracy_std,cv_precision_mean,cv_precision_std,cv_recall_mean,cv_recall_std,cv_f1_mean,cv_f1_std,cv_roc_auc_mean,cv_roc_auc_std
0,Logistic Regression,0.73,0.003,0.537,0.004,0.723,0.008,0.616,0.004,0.806,0.004


# Baseline Logistic Regression Performance

The Logistic Regression baseline achieved a mean **5-fold cross-validation ROC-AUC of 0.806 ± 0.004** on the training data.

The model also achieved a mean accuracy of **0.730**, recall of **0.723**, precision of **0.537**, and F1-score of **0.616**.

The ROC-AUC score of **0.806** will be used as the baseline benchmark for comparing subsequent models. The held-out test set remains untouched and will only be used for the final evaluation of the selected model.
